In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway to be ready
    !curl --fail \
     --retry 999 \
     --retry-all-errors \
     --retry-delay 5 \
     --retry-max-time 600 \
     http://gateway:8001/api/games

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    JUST_EXPLORE = '/kaggle/input/datasets/inversion/arc-agi-3-just-explore/arc-agi-3-just-explore'
    WORKING_COPY = '/kaggle/working/arc-agi-3-just-explore'

    # Copy the just-explore repo to a writable location
    !cp -r {JUST_EXPLORE} {WORKING_COPY}

    # Write a trimmed __init__.py that only imports HeuristicAgent
    # (the original eagerly imports langgraph, smolagents, etc. which aren't available)
    with open(f'{WORKING_COPY}/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv

from .agent import Agent, Playback
from .recorder import Recorder
from .swarm import Swarm
from .heuristic_agent import HeuristicAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "heuristicagent": HeuristicAgent,
}

# add all the recording files as valid agent names
for rec in Recorder.list():
    AVAILABLE_AGENTS[rec] = Playback
""")

    # Write a .env that routes everything through the gateway
    with open(f'{WORKING_COPY}/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
RECORDINGS_DIR=/kaggle/working/server_recording
DEBUG=False
""")

    !cd {WORKING_COPY} && \
        MPLBACKEND=agg \
        python main.py --agent heuristicagent

In [ ]:
# Non-rerun mode: produce a dummy submission so the notebook
# has valid output for the initial commit / save.
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame({'task_id': ['dummy'], 'output': ['[[0]]']}).to_parquet(
        'submission.parquet', index=False)
    print("Non-competition mode: wrote dummy submission.parquet")